In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, RobustScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
from sklearn.ensemble import IsolationForest
import plotly.express as px
import shap

import warnings
warnings.filterwarnings('ignore')

import sys
import os

sys.path.insert(0, os.path.abspath(".."))
from scaling import scale_features

from config import load_data, FEATURES, SKEWED, FRAUD_IDS

### 1. використовую усі фічі, без жодних обмежень по активності користувача  
Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.1429, Recall = 0.0000

In [6]:
df, client_data = load_data(activity_state = 1)
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first',
    'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
recall = top_20.index.isin(FRAUD_IDS)
table = pd.DataFrame({
    'score': top_20.values,
    'recall': recall
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, Recall = {recall.mean():.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.1429, Recall = 0.0000


,score,recall
person_id,,
16052012,-0.056293,False
6598101,-0.055645,False
11721056,-0.043871,False
12508879,-0.042075,False
11684714,-0.042069,False
11730290,-0.040440,False
11750504,-0.036019,False
11663876,-0.033753,False
11129711,-0.032118,False


### 2. використовую усі фічі, але додаю activity_state=2 (клієнт замовляв 3+ рази), а також дні відвідувань 3+
Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2500, Recall = 0.0000

In [9]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first',
    'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    'num_of_trn',
    'days_visits',
    'gross_amount_mean',
    'gross_amount_sum',
    'bonuses_accum_sum',
    'bonuses_used_sum',
    'num_of_waiters',
    'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
recall = top_20.index.isin(FRAUD_IDS)
table = pd.DataFrame({
    'score': top_20.values,
    'recall': recall
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, Recall = {recall.mean():.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2500, Recall = 0.0000


,score,recall
person_id,,
6598101,-0.064189,False
16052012,-0.050535,False
13995241,-0.046093,False
12508879,-0.044359,False
13074462,-0.042389,False
12715006,-0.042183,False
12270045,-0.040474,False
11790326,-0.039402,False
12547600,-0.039371,False


In [13]:
fraud_ids = client_data[client_data.index.isin(top_20.index)].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,num_of_trn,days_visits,gross_amount_mean,gross_amount_sum,bonuses_accum_sum,bonuses_used_sum,num_of_waiters,gross_amount_max,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_bonus_trn,share_bonus_after_first,num_of_places,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,,,,,,,,,,,
6598101,-0.0642,99.97,98.98,53.93,99.99,100.00,100.00,97.19,99.85,18.86,54.34,33.65,0.23,99.97,30.93,99.82,97.42,0.00,85.39,61.08
16052012,-0.0505,99.90,93.45,54.30,99.97,99.97,99.96,0.00,97.95,13.37,13.39,5.98,2.81,100.00,99.26,50.01,43.27,0.00,85.39,91.86
13995241,-0.0461,10.00,0.00,1.13,0.65,0.16,21.38,0.00,1.17,1.18,29.29,20.23,24.64,47.68,99.26,40.51,48.51,0.00,85.39,91.86
12508879,-0.0444,100.00,99.95,5.82,99.99,99.99,99.99,92.98,36.45,66.92,11.37,17.91,4.10,99.63,20.32,99.82,97.44,14.61,85.34,58.82
13074462,-0.0424,99.56,99.80,0.11,70.57,0.43,15.73,0.74,0.04,50.82,47.00,29.18,29.57,0.00,99.25,14.87,14.96,0.00,85.39,91.86
12715006,-0.0422,10.00,0.00,0.07,0.04,0.13,0.00,0.00,0.02,1.73,11.25,18.60,28.40,47.68,99.26,0.00,0.00,0.00,85.39,41.94
12270045,-0.0405,21.16,0.00,0.26,0.37,1.23,16.55,0.00,0.18,12.77,11.73,4.37,58.44,70.82,99.26,91.00,93.93,0.00,85.39,91.86
11790326,-0.0394,21.16,25.68,0.02,0.03,0.57,16.96,0.00,0.19,3.86,41.17,26.23,28.46,40.04,99.26,31.57,35.61,0.00,85.39,91.86
12547600,-0.0394,96.16,97.70,0.05,21.02,0.16,15.46,0.00,0.15,49.27,38.94,28.86,40.22,26.53,99.26,14.93,15.03,0.00,85.39,91.86


з таблиці зверху видно, що в основному картки отримують нижчий скор через аномально малі/низькі суми чеків, накобичених бофонів і тд. незважаючи на заскейлені дані, ці фічі беруться пріоритетніше

### 3. відкидаю певні фічі, а саме більшість абсолютних метрик  
- num_of_trn
- days_visits
- gross_amount_sum
- bonuses_accum_sum
- bonuses_used_sum
- num_of_waiters
- gross_amount_max
- num_of_places  

за рахунок цих метрик багато карток афектяться як аномалії, хоча насправді це можуть бути старі клієнти з великою базою транзакцій. для них фічі по типу великих сум чеків, к-ті транзакцій і тд будуть пріоритетнішими за рейтові метрики, хоча в даному контексті вони цінності не несуть.  

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2143, Recall = 0.0500

In [14]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    # 'num_of_trn',
    # 'days_visits',
    'gross_amount_mean',
    # 'gross_amount_sum',
    # 'bonuses_accum_sum',
    # 'bonuses_used_sum',
    # 'num_of_waiters',
    # 'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    'share_bonus_trn',
    'share_bonus_after_first',
    # 'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    # 'num_of_trn',
    # 'days_visits',
    'gross_amount_mean',
    # 'gross_amount_sum',
    # 'bonuses_accum_sum',
    # 'bonuses_used_sum',
    # 'num_of_waiters',
    # 'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    # 'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
recall = top_20.index.isin(FRAUD_IDS)
table = pd.DataFrame({
    'score': top_20.values,
    'recall': recall
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, Recall = {recall.mean():.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2143, Recall = 0.0500


,score,recall
person_id,,
12231243,-0.051868,False
11667048,-0.050155,False
16234805,-0.047609,False
15595950,-0.046416,False
12380638,-0.046385,False
11876561,-0.042187,False
11962560,-0.041730,False
13099115,-0.041467,False
16846666,-0.040801,True


In [15]:
fraud_ids = client_data[client_data.index.isin(top_20.index)].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_bonus_trn,share_bonus_after_first,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,,,
12231243,-0.0519,4.45,2.47,5.51,21.43,13.90,90.37,99.26,97.01,96.64,85.39,91.86
11667048,-0.0502,10.16,5.62,13.85,2.83,1.01,100.00,99.12,15.38,15.43,85.39,91.57
16234805,-0.0476,2.20,5.23,13.08,2.42,0.92,90.37,98.96,97.01,96.64,85.39,91.57
15595950,-0.0464,23.22,3.66,9.93,15.29,15.64,92.87,99.26,98.85,97.29,85.39,91.86
12380638,-0.0464,47.17,0.93,35.94,23.53,26.53,0.00,99.26,99.83,97.44,85.39,91.86
11876561,-0.0422,5.23,0.98,37.19,23.59,26.63,0.00,99.26,94.00,97.44,85.39,91.86
11962560,-0.0417,1.20,1.55,41.84,24.50,28.11,0.00,99.26,94.00,97.44,85.39,91.86
13099115,-0.0415,4.87,26.08,68.98,47.89,4.26,82.21,99.26,99.37,97.44,85.39,91.86
16846666,-0.0408,19.60,6.38,28.74,9.59,7.18,99.96,99.26,17.34,17.06,85.39,91.86


In [16]:
fraud_ids = client_data[client_data['is_fraud'] == 1].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_bonus_trn,share_bonus_after_first,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,,,
16846666,-0.0408,19.60,6.38,28.74,9.59,7.18,99.96,99.26,17.34,17.06,85.39,91.86
16376555,-0.0233,38.66,9.86,20.24,5.98,4.85,99.94,99.26,31.24,28.31,85.39,91.86
16794470,-0.0149,43.45,6.51,27.82,10.54,9.49,99.91,99.26,54.63,48.00,85.39,91.86
16921722,-0.0134,50.79,5.73,19.78,26.43,4.88,99.88,99.26,48.15,35.51,85.39,91.86
11968000,-0.0015,92.63,18.12,2.43,0.32,11.00,99.07,98.95,19.16,18.56,81.07,0.00
11970409,-0.0012,95.11,7.19,9.70,1.16,12.34,98.34,98.81,26.19,26.72,79.71,0.00
12396334,0.0135,90.51,7.18,19.17,9.99,19.72,81.85,99.15,15.32,15.43,84.70,0.00
11766135,0.0138,65.54,92.88,70.16,74.29,9.93,96.13,99.18,25.85,23.82,85.39,91.72
13969910,0.0287,37.47,36.26,1.51,0.62,6.38,99.38,94.83,25.83,23.78,85.32,88.52


### 4. відкидаю фічу share_bonus_trn
порівнюючи дві таблиці зверху бачимо, що аномаліями детектяться переважно картки, де висока частка share_bonus_trn. в той час як на реальних фрод транзакціях це не так однозначно. половина карток не так сильно відхиляється саме по share_bonus_trn

top20 recall 5% -> 20% 

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2143, Recall = 0.2000

In [17]:
df, client_data = load_data(activity_state = 2)
client_data = client_data[client_data['days_visits'] > 2]
X_std = scale_features(data=client_data, scaler_type="standard")

features = [
    # 'num_of_trn',
    # 'days_visits',
    'gross_amount_mean',
    # 'gross_amount_sum',
    # 'bonuses_accum_sum',
    # 'bonuses_used_sum',
    # 'num_of_waiters',
    # 'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    'share_top_waiter',
    # 'share_bonus_trn',
    # 'share_bonus_after_first',
    # 'num_of_places',
    'share_top_places',
    'share_bonuses_used_top_waiter'
]
skewed = [
    # 'num_of_trn',
    # 'days_visits',
    'gross_amount_mean',
    # 'gross_amount_sum',
    # 'bonuses_accum_sum',
    # 'bonuses_used_sum',
    # 'num_of_waiters',
    # 'gross_amount_max',
    'first_last_trn_diff',
    'first_second_trn_diff',
    'first_third_trn_diff',
    'time_between_trn_median',
    'trn_per_day',
    # 'num_of_places'
]

X_std2 =  scale_features(
        data = client_data,
        scaler_type = "standard",
        features = features,
        skewed = skewed,
        show_charts = False,
        fit_data = None,
)
iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,
    random_state=42
)

train_data = X_std2[~X_std2.index.isin(FRAUD_IDS)]
iso.fit(train_data)

client_data['anomaly_score_std'] = iso.decision_function(X_std2)
client_data['anomaly_label_std'] = iso.predict(X_std2)
hit_rate = client_data[client_data['is_fraud']]["anomaly_label_std"].value_counts(normalize=True).get(-1, 0)

top_20 = client_data['anomaly_score_std'].sort_values().head(20)
recall = top_20.index.isin(FRAUD_IDS)
table = pd.DataFrame({
    'score': top_20.values,
    'recall': recall
}, index=top_20.index)
print(f"Isolation Forest with contamination=0.001: Anomaly Hit Rate = {hit_rate:.4f}, Recall = {recall.mean():.4f}")
table

Isolation Forest with contamination=0.001: Anomaly Hit Rate = 0.2143, Recall = 0.2000


,score,recall
person_id,,
11808945,-0.065705,False
11667048,-0.055616,False
11972782,-0.052143,False
16846666,-0.049204,True
16376555,-0.049068,True
16794470,-0.046373,True
16062222,-0.046014,False
16921722,-0.044722,True
12720938,-0.044040,False


In [18]:
fraud_ids = client_data[client_data.index.isin(top_20.index)].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,
11808945,-0.0657,0.59,1.46,20.14,10.45,16.14,90.37,99.26,85.39,91.86
11667048,-0.0556,10.16,5.62,13.85,2.83,1.01,100.00,99.12,85.39,91.57
11972782,-0.0521,3.06,3.64,18.40,11.80,6.70,97.63,99.18,83.84,91.86
16846666,-0.0492,19.60,6.38,28.74,9.59,7.18,99.96,99.26,85.39,91.86
16376555,-0.0491,38.66,9.86,20.24,5.98,4.85,99.94,99.26,85.39,91.86
16794470,-0.0464,43.45,6.51,27.82,10.54,9.49,99.91,99.26,85.39,91.86
16062222,-0.0460,0.15,10.80,8.82,11.92,23.58,76.32,99.26,85.39,91.86
16921722,-0.0447,50.79,5.73,19.78,26.43,4.88,99.88,99.26,85.39,91.86
12720938,-0.0440,0.22,11.85,25.18,7.28,23.03,70.54,99.25,85.25,91.86


In [19]:
fraud_ids = client_data[client_data['is_fraud'] == 1].index

# Sort fraud_ids by anomaly score (ascending: most anomalous first)
fraud_ids_sorted = client_data.loc[fraud_ids, 'anomaly_score_std'].sort_values().index

# Collect percentiles for each fraud person_id (for selected features)
train = X_std2[~X_std2.index.isin(FRAUD_IDS)]

features_for_table = ['anomaly_score_std'] + features
rows = []

for person_id in fraud_ids_sorted:
    row = {}
    for feat in features_for_table:
        if feat == 'anomaly_score_std':
            row[feat] = round(client_data.loc[person_id, 'anomaly_score_std'], 4)
        else:
            val = X_std2.loc[person_id, feat]
            # Calculate percentile vs train
            percentile = (train[feat] < val).mean() * 100
            row[feat] = round(percentile, 2)
    rows.append(row)

result_df = pd.DataFrame(rows, index=fraud_ids_sorted)
result_df.index.name = "person_id"
result_df.to_csv('res_df.csv')
result_df

,anomaly_score_std,gross_amount_mean,first_last_trn_diff,first_second_trn_diff,first_third_trn_diff,time_between_trn_median,trn_per_day,share_top_waiter,share_top_places,share_bonuses_used_top_waiter
person_id,,,,,,,,,,
16846666,-0.0492,19.60,6.38,28.74,9.59,7.18,99.96,99.26,85.39,91.86
16376555,-0.0491,38.66,9.86,20.24,5.98,4.85,99.94,99.26,85.39,91.86
16794470,-0.0464,43.45,6.51,27.82,10.54,9.49,99.91,99.26,85.39,91.86
16921722,-0.0447,50.79,5.73,19.78,26.43,4.88,99.88,99.26,85.39,91.86
11970409,-0.0066,95.11,7.19,9.70,1.16,12.34,98.34,98.81,79.71,0.00
11968000,-0.0000,92.63,18.12,2.43,0.32,11.00,99.07,98.95,81.07,0.00
11766135,0.0027,65.54,92.88,70.16,74.29,9.93,96.13,99.18,85.39,91.72
12396334,0.0250,90.51,7.18,19.17,9.99,19.72,81.85,99.15,84.70,0.00
13969910,0.0397,37.47,36.26,1.51,0.62,6.38,99.38,94.83,85.32,88.52
